<a href="https://colab.research.google.com/github/AnithaN21/AI-12-days-Bootcamp/blob/main/Day2_ResumeExtractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 2 — JSON Résumé Extractor (Gemini + Pydantic)**Goal:** Force Gemini into Pydantic-validated JSON. Handle the 3 most common failures.**Time:** 90 minutes.

## Step 1 — Install + key

In [5]:
import os
import getpass

if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key:')

Gemini API key:··········


## Step 2 — Define the Pydantic model8 fields. Use `Optional` for fields that may be missing — phone is the most common.

In [7]:
from pydantic import BaseModel
from typing import List, Optional

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float

## Step 3 — Extract function with retry on schema failure

In [8]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:
    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'Extract a Resume JSON from this text. Return ONLY JSON, no markdown.\n\n{raw_text}',
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)
        except ValidationError as e:
            if attempt == max_retries:
                raise
            # Retry once with the broken JSON in the prompt
            fix_prompt = (f'Fix this JSON to match schema. Errors: {e}. '
                          f'Original: {resp.text}')
            resp = client.models.generate_content(
                model='gemini-2.5-flash', contents=fix_prompt,
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)

## Step 4 — Run on 3 sample résumésLoad sample data from `../data/sample_resumes.txt` (5 anonymised résumés, blank line separated).

In [11]:
# Load sample résumés
with open('./sample_resumes.txt') as f:
    resumes = [r.strip() for r in f.read().split('---') if r.strip()]

results = []
for i, r in enumerate(resumes[:3]):
    try:
        parsed = extract_resume(r)
        results.append(parsed)
        print(f'Résumé {i+1}: {parsed.name} — {len(parsed.skills)} skills')
    except Exception as e:
        print(f'Résumé {i+1}: FAILED — {e}')

print('\n', results[0].model_dump_json(indent=2) if results else 'no results')

Résumé 1: Ravi Kumar — 6 skills
Résumé 2: Sneha Reddy — 6 skills
Résumé 3: Arun Pillai — 8 skills

 {
  "name": "Ravi Kumar",
  "email": "ravi.kumar@example.com",
  "phone": "+91-9876543210",
  "education": [
    {
      "degree": "B.Tech Computer Science",
      "institution": "Aditya University",
      "year": 2026
    },
    {
      "degree": "Intermediate",
      "institution": "Narayana Junior College",
      "year": 2022
    }
  ],
  "skills": [
    "Python",
    "Java",
    "SQL",
    "Git",
    "Linux",
    "REST APIs"
  ],
  "projects": [
    "Built a Flask REST API for college placement portal (3-week internship at startup)",
    "Solved 250+ DSA problems on LeetCode (Top 5% in college)",
    "Final-year project: ML model for crop yield prediction (Random Forest, 87% accuracy)"
  ],
  "experience_years": 0.25
}


## Step 5 — Test broken inputPass an empty string. Confirm graceful handling, not a crash.

In [34]:
try:
    bad = extract_resume('')
    print('Unexpected success:', bad.model_dump_json())
except Exception as e:
    print('Caught gracefully:', type(e).__name__)
    print('Message:', str(e)[:200])

Caught gracefully: ValueError
Message: Resume text is empty


## Acceptance- 3 résumés processed end-to-end ✓- 1 broken input handled without traceback ✓- README documents which 3 errors you handled and how